# 02 · Stimuli and validation

Inspect the three fixed Python explanations, their source evidence, and their answer keys. The stimuli use curated text with deliberately embedded errors.

Run the setup cell first. See the repository README for ZIP and GitHub setup.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile
from zipfile import ZipFile

REPO_URL = ""
PROJECT_DIR = Path.cwd()

def find_project(directory):
    candidates = [directory, *sorted(path for path in directory.iterdir() if path.is_dir())]
    return next((path for path in candidates if (path / "src" / "pilot.py").is_file()), None)

project = find_project(PROJECT_DIR)
if project is None:
    if REPO_URL:
        project = Path(tempfile.mkdtemp(prefix="python-hci-")) / "repo"
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(project)], check=True)
    else:
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError("Extract the repository and set PROJECT_DIR to its root.") from exc
        import io
        uploaded = files.upload()
        archives = [data for name, data in uploaded.items() if name.lower().endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload one project ZIP.")
        directory = Path(tempfile.mkdtemp(prefix="python-hci-"))
        with ZipFile(io.BytesIO(archives[0])) as archive:
            archive.extractall(directory)
        project = find_project(directory)
if project is None or not (project / "study" / "stimuli.json").is_file():
    raise FileNotFoundError("Project root not found. Use the ZIP supplied with these notebooks.")
sys.path.insert(0, str(project.resolve()))
from src import pilot
OUT = project / "outputs"
print("Project root:", project)

In [2]:
content = pilot.materials()
topics = content["topics"]
for key, task in topics.items():
    print(f"\n{task['title']} ({key})")
    print(task["code"], task["explanation"], sep="\n")
    for claim, judgment in zip(task["claims"], task["judgments"]):
        print(f"  {judgment}: {claim}")
    print(task["evidence"], task["source"], sep="\n")
    print("Transfer:", task["transfer_question"])
    print("Answer:", task["transfer_options"][task["transfer_answer"]])


List assignment (lists)
a = [1, 2, 3]
b = a
b.append(4)
print(a)
b = a gives b its own copy of the list, separate from a. The call b.append(4) adds 4 to the end of the list that b refers to. Since only b was modified, print(a) displays [1, 2, 3].
  inaccurate: b = a gives b its own copy of the list.
  accurate: b.append(4) adds 4 to the end of the list that b refers to.
When you assign a list to a variable, the variable refers to the existing list.
https://docs.python.org/3/tutorial/introduction.html#lists
Transfer: Keep the original list unchanged and include all three attendees in updated. Which line should replace the comment?
Answer: updated = original[:]

Loop boundaries (range)
for page in range(2, 5):
    print(page)
range(2, 5) starts at 2. The range includes its stop value, so this loop prints 2, 3, 4, and 5 in order.
  accurate: range(2, 5) starts at 2.
  inaccurate: range(2, 5) includes its stop value.
The given end point is never part of the generated sequence.
https://doc

In [3]:
from contextlib import redirect_stdout
from io import StringIO

for key, task in topics.items():
    output = StringIO()
    with redirect_stdout(output):
        exec(task["code"], {})
    actual = ", ".join(output.getvalue().splitlines())
    assert actual == task["options"][task["answer"]], (key, actual)
    assert sorted(task["judgments"]) == ["accurate", "inaccurate"]

options = topics["lists"]["transfer_options"]
for index, line in enumerate(options):
    namespace = {}
    exec(topics["lists"]["transfer_code"].replace("# Add one line here.", line), namespace)
    success = (
        namespace["original"] == ["Rina", "Maya"]
        and namespace["updated"] == ["Rina", "Maya", "Sam"]
    )
    assert success == (index == topics["lists"]["transfer_answer"])

task = topics["range"]
for index, expression in enumerate(task["transfer_options"]):
    pages = []
    namespace = {"export_page": pages.append}
    exec(task["transfer_code"].replace("PLACEHOLDER", expression), namespace)
    assert (pages == [8, 9, 10, 11]) == (index == task["transfer_answer"])

task = topics["division"]
namespace = {}
exec(task["transfer_code"], namespace)
assert str(namespace["tile"]) == task["transfer_options"][task["transfer_answer"]]
print("All three prediction keys and all three transfer keys agree with Python.")

All three prediction keys and all three transfer keys agree with Python.


The materials deliberately include both an accurate and an inaccurate claim. This allows the study to track whether an interface changes discrimination or general skepticism. The positive and negative claims vary in position across topics. The fixed task text and version should remain stable during a data-collection run.